Mechanistical Interpretability = toolkit to explore how models work

Neurons:
- monosemantic neurons = carry one feature
- multisemantic neurons = carry a blend of many features

# Observation methods

### Probing (2019)
[[paper]](https://aclanthology.org/N19-1419.pdf)
<br>__Idea:__ train a lightweight classifier to test whether a property (POS tags, syntax depth, etc.) is linearly readable from hidden states 

### Logit Lens (2020)
[[paper]](https://arxiv.org/pdf/2303.08112)
<br> __Idea:__ At each layer project intermediate outputs into vocab space using fixed final layer projection
<br>Motivation = to see “the model’s current guess” at each layer

### Tuned Lens (2023)
[[paper]](https://arxiv.org/pdf/2303.08112)
<br>__Idea:__ The same as LogitLens but they add additional trainable layer before each projection. It is trained as adapter on a frozen model
<br>Motivation = to see “what model could potentially predict at each layer”

### Direct Logit Attribution
[[paper]](https://arxiv.org/pdf/2303.08112)
<br>__Idea:__ Linearly attribute final logits to contributions from specific components (heads/MLPs/positions); useful “who pushed the logit?” readout, but be cautious about LayerNorm/linearization assumptions and adversarial edge cases

### Attention Head Analysis (2022)
[[paper]](https://arxiv.org/pdf/2211.00593)
<br>Inspect attention maps & head roles (e.g., induction heads, name-mover heads) and how they compose into circuits on concrete tasks like IOI

### Representation Similarity (1997)
[[paper]](https://proceedings.mlr.press/v97/kornblith19a/kornblith19a.pdf)
<br>A family of methods to compare layer outputs (i.e. SVCCA/CKA/RSA)

Comparison can be done:
- between different layers (across model depth)
- between corresponding layers of two models
- between model checkpoints (across training completeness)

It allows to see the signal dynamics / possible layer redundancy / possible phase shift in training process etc

### Patchscopes (2024)
[[paper]](https://arxiv.org/abs/2401.06102)
<br>Idea: copy-paste weight to another model and see how it affects the answer

Algorithm:
- do "source" run
- prepare target run: construct a prompt with a placeholder
- copy-paste inspected weight to the desired position (layer, placeholder)
- do target run
- analyze what was generated

### Sparse Autoencoders (2024)
[[paper]](https://arxiv.org/abs/2406.04093)
<br>__Idea:__ try to build a sparse representation of intermediate signal - patterns

Algorithm:
- choose the layer to inspect
- freeze the model and train SAE
- intermediate state of SAE represents sparse “features” that fire on monosemantic concepts

Top-K<br>
Dictionary Learning



Some patterns in Attention Heads

| Pattern Name                   | What It Looks Like                                                                         | Likely Function                                                                                         |
| ------------------------------ | ------------------------------------------------------------------------------------------ | ------------------------------------------------------------------------------------------------------- |
| **Induction Heads**            | Strong diagonal attention from a token to its *previous occurrence* earlier in the context | Enables “copy after repetition” — a mechanism for few-shot learning and in-context pattern continuation |
| **Duplicate Token Heads**      | Attend to the *nearest identical token* in the past                                        | Support repetition and exact match completion                                                           |
| **Name Mover Heads**           | In question-answer tasks, jump from the question name to the answer name position          | Important in IOI (Indirect Object Identification) circuits                                              |
| **Next Token Heads**           | Attend to the *immediately previous token*                                                 | Acts like a basic Markov chain, helping next-token prediction in local contexts                         |
| **Syntactic Heads**            | Focus on fixed dependency relations (e.g., subject → verb, noun → determiner)              | Encodes grammatical structure                                                                           |
| **Bridging Heads**             | Attend from a pronoun or anaphor to its referent                                           | Supports coreference resolution                                                                         |
| **Positional Heads**           | Almost entirely determined by relative positions, regardless of token identity             | Often act as position encoders or help combine local context windows                                    |
| **Stopword Suppression Heads** | Attend away from common stopwords                                                          | Help downstream components ignore low-information tokens                                                |
| **Translation Heads**          | In multilingual models, attend to semantically equivalent words in different languages     | Assist in cross-lingual alignment                                                                       |
| **Copy Heads**                 | Nearly uniform attention to a specific previous position                                   | Support literal copying of subsequences                                                                 |


# Intervention methods

### Activation Patching (2020+)  
[[paper]](https://arxiv.org/abs/2005.00397)  
**Idea:** Swap activations between a “clean” run (correct behavior) and a “corrupted” run (wrong behavior) at specific locations. If patching restores performance, that component is causally important

Synonyms: Causal Tracing, Resample Ablation, Interchange Intervention, Causal Mediation Analysis

**Algorithm:**  
- Prepare clean and corrupted runs.  
- At a target layer/head/position, replace corrupted activation with the clean one.  
- Compare outputs to assess recovery.  

**Goal:** Localize necessary/sufficient components for a behavior.

---

### Attribution Patching (2023)  
[[paper]](https://arxiv.org/abs/2304.11489)  
**Idea:** Approximate patching importance for all components using two forward passes and one backward pass, producing a score without full enumeration.  
**Goal:** Efficiently pre-filter candidates for full activation patching.

---

### Path Patching (2023)  
[[paper]](https://arxiv.org/abs/2306.17806)  
**Idea:** Patch activations only along a specific hypothesized edge (source → target), not the whole node.  
**Goal:** Distinguish direct effects from mediated effects and refine circuit maps.

---

### Causal Scrubbing (2023)  
[[paper]](https://arxiv.org/abs/2304.03429)  
**Idea:** Formal framework to test whether a hypothesized circuit fully explains a behavior by resampling inputs in a behavior-preserving way.  
**Goal:** Validate sufficiency of proposed mechanisms.

---

### Knowledge Editing (2022–2023)  
[[ROME]](https://arxiv.org/abs/2202.05262)  
[[MEMIT]](https://arxiv.org/abs/2302.12464)  
**Idea:** Change specific factual associations by editing MLP weights.  
- ROME: Rank-one update for single facts.  
- MEMIT: Multi-fact editing.  
**Goal:** Locate stored knowledge and study edit propagation.

---

### Component Ablations (2019+)  
[[paper]](https://arxiv.org/abs/1906.03731)  
**Idea:** Zero out or remove components (attention heads, MLPs, neurons) and measure performance drop.  
**Goal:** Measure necessity of components for specific capabilities.

---

### Causal Mediation (2020+)  
[[paper]](https://proceedings.neurips.cc/paper/2020/hash/7ae447d3e5d6cfa6b3ebf2c3d05b3a8e-Abstract.html)  
**Idea:** Estimate how much of the effect from input to output flows through a given mediator.  
**Goal:** Separate direct vs. indirect contributions of components.

---

### Activation Steering (2023–2024)  
[[paper]](https://arxiv.org/abs/2302.12173)  
[[SAE-guided]](https://arxiv.org/abs/2406.04093)  
**Idea:** Add a learned or contrastive “steering vector” to the residual stream at inference to shift behavior.  
**Variants:** Contrastive Activation Addition, SAE-feature steering.  
**Goal:** Test causal influence of directions/features and achieve targeted generation control.


# Observation methods

### Probing (2019)  
[[paper]](https://aclanthology.org/N19-1419.pdf)  
**Idea:** Train a lightweight classifier to test whether a property (POS tags, syntax depth, etc.) is linearly readable from hidden states.  
**Goal:** Check whether certain information is *present* in representations.  
**Notes:** High probe accuracy does not necessarily mean the model *uses* that information.

---

### Logit Lens (2020)  
[[paper]](https://arxiv.org/pdf/2303.08112)  
**Idea:** At each layer, project intermediate outputs into vocabulary space using the model’s fixed final projection (LM head).  
**Goal:** See “the model’s current guess” at each layer without extra training.

---

### Tuned Lens (2023)  
[[paper]](https://arxiv.org/pdf/2303.08112)  
**Idea:** Same as Logit Lens, but add a small trainable affine transformation before projection. Trained on a frozen model to align intermediate representations with the final-layer readout space.  
**Goal:** See “what the model could potentially predict” at each layer, with cleaner and more accurate intermediate predictions.

---

### Direct Logit Attribution (2023)  
[[paper]](https://arxiv.org/pdf/2303.08112)  
**Idea:** Decompose final logits into additive contributions from each component (attention head, MLP, position) using the residual stream’s additive structure.  
**Goal:** Identify “who pushed the logit” and by how much.  
**Notes:** Approximations may be needed due to LayerNorm; results are direct effects only.

---

### Attention Head Analysis (2022)  
[[paper]](https://arxiv.org/pdf/2211.00593)  
**Idea:** Inspect attention maps for individual heads to find recurring patterns (e.g., induction heads, name movers, syntactic heads).  
**Goal:** Classify heads by function and locate potential circuit components.

---

### Representation Similarity (1997 / 2019)  
[[paper]](https://proceedings.mlr.press/v97/kornblith19a/kornblith19a.pdf)  
**Idea:** Compare layer representations using SVCCA, PWCCA, CKA, or RSA to detect structural similarities or changes.  
**Can compare:**  
- Layers within the same model  
- Corresponding layers between models  
- Checkpoints during training  
**Goal:** Identify redundancy, phase shifts, or convergence patterns.

---

### Patchscopes (2024)  
[[paper]](https://arxiv.org/abs/2401.06102)  
**Idea:** Copy an activation from a source run into a target run with a prompt designed to elicit interpretation from the model.  
**Algorithm:**  
- Run the model on the source input and capture the activation.  
- Create a target prompt with a placeholder for the activation.  
- Insert (patch) the source activation into the target run.  
- Observe the generated output to interpret the activation’s meaning.

---

### Sparse Autoencoders (2024)  
[[paper]](https://arxiv.org/abs/2406.04093)  
**Idea:** Train a sparse autoencoder on frozen model activations to discover features that fire on monosemantic concepts.  
**Algorithm:**  
- Choose a layer to inspect.  
- Freeze the base model and collect activations.  
- Train an SAE to reconstruct activations from a sparse feature code.  
- Inspect features by their top-activating tokens/contexts.  
**Variants:** Top-K SAEs, orthogonal regularization.  
**Goal:** Build a dictionary of interpretable features for later causal testing.

---